# $\tau$ convergence at $\omega = 0.1$

Checking how big the Trotter step can get before results change. Same run for every job ($L = 24$, OBC, $m = 400$, 4 periods, $g = 0.2, 1.0$, both parities), just different $\tau = T/N$ with $T = 2\pi/\omega$ and $N$ = `steps_per_period`. $N = 1280$ ($\tau \approx 0.049$, same as production) is the reference.

Data folder `tebd_tau_convergence_2026-09` goes next to this notebook.

`tol` is the max error I'll accept. `restrict_to_unsaturated = True` only compares times before the reference hits $m$.

In [ ]:
import os
import re
import glob
import numpy as np
import h5py
import matplotlib.pyplot as plt

root  = "tebd_tau_convergence_2026-09"
tol   = 1e-2
restrict_to_unsaturated = False
chans = {"x": r"$\langle X(t)X(0)\rangle$", "y": r"$\langle Y(t)Y(0)\rangle$",
         "z": r"$\langle Z(t)Z(0)\rangle$", "e": r"$\langle H(t)\rangle/L$"}
job_re   = re.compile(r"g(-?[\d.]+)_site(\d+)_p(even|odd)_sp(\d+)")
batch_re = re.compile(r"L(\d+)_\w+?_m(\d+)")

Loads one `output.h5`: times, bond dims, $T$, the three autocorrelations (re + im), and $\langle H(t)\rangle/L$.

In [ ]:
def load(fname):
    with h5py.File(fname, "r") as f:
        d = {c: f[f"re_autoc_{c}"][()] + 1j * f[f"im_autoc_{c}"][()] for c in "xyz"}
        d["e"] = f["energy_t"][()] / f["L"][()]
        d["t"], d["chi"], d["T"] = f["times"][()], f["maxdims"][()], 2 * np.pi / f["omega"][()]
    return d

$t_\text{sat}$ = first time the reference's bond dim hits $m$. After that truncation error mixes in. Returns $\infty$ if it never hits or the MPS is exact.

In [ ]:
def t_saturation(rs, batch):
    L, m = map(int, batch_re.search(batch).groups())
    if m >= 2 ** (L // 2):
        return np.inf
    d = rs[max(rs)]
    return d["t"][np.argmax(d["chi"] >= m)] if (d["chi"] >= m).any() else np.inf

Compares each $\tau$ to the reference ($\tau\approx0.049$ from $1280$ steps per period). Grids are nested, so coarse point $i$ lines up with reference point $i\,r$, $r = N_\text{ref}/N$.

If $A_\tau = A_\text{exact} + C\tau^2$ then $A_\tau - A_\text{ref} = C(\tau^2 - \tau_\text{ref}^2)$, so the actual error is
$$\varepsilon(\tau) = \max_t |A_\tau - A_\text{ref}|\,\frac{\tau^2}{\tau^2 - \tau_\text{ref}^2}.$$

Observed order from neighboring steps, $D(\tau) = \max_t |A_\tau - A_{\tau/2}|$:
$$p = \log_2\frac{D(\tau)}{D(\tau/2)},$$
should be about 2.

In [ ]:
def compare(rs, t_win):
    sps = sorted(rs, reverse=True)
    ref, T = rs[sps[0]], rs[sps[0]]["T"]
    tau_ref = T / sps[0]
    D, out = {}, {}
    for fine, sp in zip(sps, sps[1:]):
        d, tau = rs[sp], T / sp
        mask = d["t"] <= t_win
        run_err = {c: np.maximum.accumulate(np.abs(d[c] - ref[c][:: sps[0] // sp])) for c in chans}
        D[sp] = {c: np.abs(d[c] - rs[fine][c][:: fine // sp])[mask].max() for c in chans}
        out[sp] = {"tau": tau, "t": d["t"], "run_err": run_err,
                   "eps": {c: run_err[c][mask][-1] * tau**2 / (tau**2 - tau_ref**2) for c in chans}}
        if fine in D:
            out[sp]["order"] = {c: np.log(D[sp][c] / D[fine][c]) / np.log(fine / sp) for c in chans}
    return out

Load everything. Unfinished jobs have no `output.h5` and get skipped.

In [ ]:
runs = {}
for fname in sorted(glob.glob(os.path.join(root, "*", "*", "output.h5"))):
    batch, job = fname.split(os.sep)[-3:-1]
    g, _, p, sp = job_re.search(job).groups()
    runs.setdefault((batch, float(g), p), {})[int(sp)] = load(fname)

Run the comparison for each $(g, \text{parity})$.

In [ ]:
t_sats  = {k: t_saturation(rs, k[0]) for k, rs in runs.items()}
t_wins  = {k: t_sats[k] if restrict_to_unsaturated else np.inf for k in runs}
results = {k: compare(rs, t_wins[k]) for k, rs in sorted(runs.items())}
batches = sorted({k[0] for k in results})
g_vals  = sorted({k[1] for k in results})

Error table. $\varepsilon$ for each quantity plus observed order $p$. Error of production $\tau$ itself is about $\varepsilon(0.098)/4$.

In [ ]:
print(f"{'batch':38s} {'g':>4s} {'par':>4s} {'tau':>7s}  " + "  ".join(f"eps_{c}   " for c in chans) + "  order x y z e")
for (b, g, p), res in results.items():
    T = runs[(b, g, p)][max(runs[(b, g, p)])]["T"]
    print(f"{b:38s} {g:4.1f} {p:>4s}  window t <= {min(t_wins[(b, g, p)], res[min(res)]['t'][-1]) / T:.2f} T, reference hits maxdim at t = {t_sats[(b, g, p)] / T:.2f} T")
    for sp, r in res.items():
        order = " ".join(f"{r['order'][c]:4.2f}" for c in chans) if "order" in r else ""
        print(f"{'':48s} {r['tau']:7.4f}  " + "  ".join(f"{r['eps'][c]:.1e}" for c in chans) + f"  {order}")

Largest $\tau$ with $\varepsilon <$ `tol` everywhere. This is 4 periods, production is 20, so scale by ~5 if error grows linearly.

In [ ]:
for b in batches:
    keys = [k for k in results if k[0] == b]
    ok = [sp for sp in results[keys[0]] if all(max(results[k][sp]["eps"].values()) < tol for k in keys)]
    print(f"{b}: largest tau with eps < {tol:g} everywhere: " +
          (f"{results[keys[0]][min(ok)]['tau']:.4f} (steps_per_period = {min(ok)})" if ok else "none"))

Raw curves vs $t/T$, even sector. Good $\tau$ should sit on the reference.

In [ ]:
fig, ax = plt.subplots(len(chans), len(g_vals), figsize=(5 * len(g_vals), 2.4 * len(chans)), sharex=True, squeeze=False)
for j, g in enumerate(g_vals):
    rs = runs[(batches[-1], g, "even")]
    for sp in sorted(rs, reverse=True):
        d = rs[sp]
        for i, c in enumerate(chans):
            ax[i, j].plot(d["t"] / d["T"], d[c].real, lw=1, label=rf"$\tau$={d['T'] / sp:.3f}")
    ax[0, j].set_title(f"{batches[-1]}, g={g}, even")
    ax[-1, j].set_xlabel(r"$t/T$")
for i, c in enumerate(chans):
    ax[i, 0].set_ylabel(chans[c] if c == "e" else "Re " + chans[c])
ax[0, 0].legend(fontsize=7)
fig.tight_layout()

$\varepsilon$ vs $\tau$, log-log. Solid even, dashed odd, dotted is $\tau^2$, gray is `tol`.

In [ ]:
fig, ax = plt.subplots(len(batches), len(g_vals), figsize=(5 * len(g_vals), 4 * len(batches)), squeeze=False)
for i, b in enumerate(batches):
    for j, g in enumerate(g_vals):
        for p, ls in [("even", "-"), ("odd", "--")]:
            res = results[(b, g, p)]
            taus = np.array([r["tau"] for r in res.values()])
            for k, c in enumerate(chans):
                ax[i, j].loglog(taus, [r["eps"][c] for r in res.values()], ls, marker="o", ms=3, color=f"C{k}", label=f"{c} {p}")
        e0 = np.median([res[max(res)]["eps"][c] for c in chans])
        ax[i, j].loglog(taus, e0 * (taus / taus.min()) ** 2, "k:", label=r"$\propto\tau^2$")
        ax[i, j].axhline(tol, color="gray", lw=0.8)
        ax[i, j].set(title=f"{b}, g={g}", xlabel=r"$\tau$", ylabel=r"$\varepsilon(\tau)$")
ax[0, 0].legend(fontsize=6, ncol=2)
fig.tight_layout()

Running max error vs $t/T$, even sector. Dotted line is $t_\text{sat}$.

In [ ]:
fig, ax = plt.subplots(len(batches), len(g_vals), figsize=(5 * len(g_vals), 3.5 * len(batches)), squeeze=False)
for i, b in enumerate(batches):
    for j, g in enumerate(g_vals):
        T = runs[(b, g, "even")][max(runs[(b, g, "even")])]["T"]
        for sp, r in results[(b, g, "even")].items():
            ax[i, j].semilogy(r["t"] / T, np.max([r["run_err"][c] for c in chans], axis=0), label=rf"$\tau$={r['tau']:.3f}")
        if np.isfinite(t_sats[(b, g, "even")]):
            ax[i, j].axvline(t_sats[(b, g, "even")] / T, color="k", ls=":")
        ax[i, j].axhline(tol, color="gray", lw=0.8)
        ax[i, j].set(title=f"{b}, g={g}, even", xlabel=r"$t/T$", ylabel="running max error vs ref")
ax[0, 0].legend(fontsize=7)
fig.tight_layout()